### Import Libraries (Google Earth Engine)

In [ ]:
from pathlib import Path

import ee
import geemap

### Authenticate Google Earth Engine

In [ ]:
ee.Authenticate()
ee.Initialize()

print("Earth Engine initialized successfully!")
print(f"EE version: {ee.__version__}")

### Region OF Interest

In [ ]:
# Create an interactive map
Map = geemap.Map(center=[58.5, -118.5], zoom=8)  # Alberta fire region

# Add a basemap
Map.add_basemap("SATELLITE")

# Display the map
Map

### Gather pre fire image collection

In [ ]:
# aoi = ee.Geometry.Rectangle([-120.5, 58.5, -118.0, 60.5])
aoi = ee.Geometry.Rectangle([-120.2, 59.0, -119.7, 59.35])
pre_fire_collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate("2023-05-01", "2023-05-31")  # Before fire season
    .filterBounds(aoi)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 2))  # Less than 20% clouds
    .select(["B4", "B3", "B2", "B8", "B12"])  # RGB and NIR and SWIR
)
print(f"Pre-fire images found: {pre_fire_collection.size().getInfo()}")

### Gather post fire image collection

In [ ]:
post_fire_collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate("2023-09-01", "2023-09-30")  # After fire containment
    .filterBounds(aoi)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 2))
)

print(f"Post-fire images found: {post_fire_collection.size().getInfo()}")

### Take median of collections

In [ ]:
pre_fire_image = pre_fire_collection.median()
post_fire_image = post_fire_collection.median()

In [ ]:
def calculate_nbr(image: ee.Image) -> ee.Image:
    """Calculate NBR using NIR (B8) and SWIR (B12) bands."""
    # NBR = (NIR - SWIR) / (NIR + SWIR)
    nir = image.select("B8")
    swir = image.select("B12")
    return nir.subtract(swir).divide(nir.add(swir)).rename("NBR")

In [ ]:
nbr_pre = calculate_nbr(pre_fire_image)
nbr_post = calculate_nbr(post_fire_image)

In [ ]:
dnbr = nbr_pre.subtract(nbr_post).rename("dNBR")

In [ ]:
dnbr_normalized = dnbr.subtract(-0.1).divide(0.9).clamp(0, 1).rename("severity")

In [ ]:
# Create interactive map
Map = geemap.Map(center=[59.2, -119.0], zoom=9)

# Visualization parameters
rgb_vis = {
    "bands": ["B4", "B3", "B2"],  # Red, Green, Blue
    "min": 0,
    "max": 3000,
    "gamma": 1.4,
}

dnbr_vis = {
    "min": -0.1,
    "max": 0.8,
    "palette": ["green", "yellow", "orange", "red", "darkred"],
}

severity_vis = {
    "min": 0,
    "max": 1,
    "palette": [
        "#2E7D32",  # Dark Green     -> unburned/healthy
        "#66BB6A",  # Light Green    -> low severity
        "#FDD835",  # Yellow         -> moderate severity
        "#FB8C00",  # Orange         -> high severity
        "#E65100",  # Dark Orange    -> very high severity
        "#BF360C",
    ],  # Dark Red       -> extreme severity
}

# Add layers to map
Map.addLayer(pre_fire_image, rgb_vis, "Pre-fire RGB (May 2023)")
Map.addLayer(post_fire_image, rgb_vis, "Post-fire RGB (Sept 2023)")
Map.addLayer(dnbr, dnbr_vis, "dNBR (raw)")
Map.addLayer(dnbr_normalized, severity_vis, "Burn Severity (0-1)")
Map.addLayer(aoi, {}, "Area of Interest", shown=False)

# Add layer control
Map.addLayerControl()

# Display map
Map

In [ ]:
if Map.user_roi:
    aoi = Map.user_roi
    print(f"✅ New AOI size: ~{aoi.area(maxError=1).divide(1e6).getInfo():.0f} km²")
else:
    print("❌ Draw a rectangle on the map first!")

### Calculate Burn Statistics

In [ ]:
# Get severity statistics
severity_stats = dnbr_normalized.reduceRegion(
    reducer=ee.Reducer.mean()
    .combine(
        reducer2=ee.Reducer.minMax(),
        sharedInputs=True,
    )
    .combine(
        reducer2=ee.Reducer.stdDev(),
        sharedInputs=True,
    ),
    geometry=aoi,
    scale=10,  # Sentinel-2 resolution
    maxPixels=1e13,
)

stats = severity_stats.getInfo()
print("\n📊 Burn Severity Statistics:")
print(f"Mean severity: {stats.get('severity_mean', 0):.3f}")
print(f"Min severity: {stats.get('severity_min', 0):.3f}")
print(f"Max severity: {stats.get('severity_max', 0):.3f}")
print(f"Std Dev: {stats.get('severity_stdDev', 0):.3f}")


# Calculate area of different severity classes
def calculate_severity_areas(severity_image: ee.Image, aoi: ee.Geometry) -> dict:
    """Calculate area (in hectares) for each severity class."""
    # Define severity thresholds (USGS classification)
    unburned = severity_image.lt(0.1)
    low = severity_image.gte(0.1).And(severity_image.lt(0.27))
    moderate_low = severity_image.gte(0.27).And(severity_image.lt(0.44))
    moderate_high = severity_image.gte(0.44).And(severity_image.lt(0.66))
    high = severity_image.gte(0.66)
    # Calculate pixel areas (in square meters)
    pixel_area = ee.Image.pixelArea()

    # Calculate areas for each class
    classes = {
        "Unburned": unburned,
        "Low Severity": low,
        "Moderate-Low Severity": moderate_low,
        "Moderate-High Severity": moderate_high,
        "High Severity": high,
    }

    results = {}
    for name, mask in classes.items():
        area = pixel_area.updateMask(mask).reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=aoi,
            scale=10,
            maxPixels=1e13,
        )
        # Convert to hectares
        area_ha = ee.Number(area.get("area")).divide(10000)
        results[name] = area_ha.getInfo()

    return results


areas = calculate_severity_areas(dnbr_normalized, aoi)

print("\n🔥 Burned Area by Severity Class:")
for severity_class, area in areas.items():
    print(f"{severity_class:25s}: {area:>10,.0f} hectares")

total_burned = sum([v for k, v in areas.items() if "Severity" in k])
print(f"{'Total Burned Area':25s}: {total_burned:>10,.0f} hectares")

In [ ]:
area_sq_meters = aoi.area().getInfo()
area_sq_km = area_sq_meters / 1_000_000

print(f"Area: {area_sq_km:,.0f} km²")

In [ ]:
post_fire_rgb = post_fire_image.select(["B4", "B3", "B2"])

In [ ]:
# Scale to uint8
post_fire_rgb_uint8 = post_fire_rgb.divide(3000).clamp(0, 1).multiply(255).uint8()

In [ ]:
output_dir = Path("./alberta_fire_data")
output_dir.mkdir(parents=True, exist_ok=True)

print("  📥 Downloading RGB image...")
geemap.download_ee_image(
    image=post_fire_rgb_uint8,
    filename=str(output_dir / "postfire_rgb.tif"),
    region=aoi,
    crs="EPSG:4326",
)

print("  📥 Downloading severity labels...")
geemap.download_ee_image(
    image=dnbr_normalized.toFloat(),
    filename=str(output_dir / "severity_label.tif"),
    scale=10,
    region=aoi,
    crs="EPSG:4326",
)

### Overview of GeoAI: Machine learning

![ide](/home/valhassa/Projects/geo-deep-learning/notebooks/gdl_overview.png)